# The baseline noise (`bstd`) of each channel, after the 450-sample window

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

`bstd-distribution.ipynb` and `baseline-window-onset.ipynb` showed that with a baseline window of samples 10 to 500 the `bstd` of the real triggers grew with the pulse height, because the rising edge of the pulse (median onset at sample 483 to 486) was inside the window. The readout of the registry now ends the window at **sample 450** (`registry/readouts.toml`, 2026-11-02), and the eight DCRC1pB-triggered series near -82 V have been reprocessed with it into the same output folder, `~/goldsluice-output/r76-82v-dcrc1pb`.

Here is the `bstd` distribution of every channel again, from the new output: real triggers (blue) against random triggers (grey), the old window (500) against the new one (450) for the real triggers, and the height against `bstd`. The notebook reads `pretrigger_samples` from the manifest of the output and keeps one cache per window, so the old and the new numbers are never mixed. The cache of the 500 window, written by `bstd-distribution.ipynb`, is used for the comparison when it exists. Run with the `darkmatter_cli_env` kernel.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from matplotlib import colors

from goldpan.dataset import Dataset

LETTERS = "ABCDEF"
_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
OUT = Path(Dataset.open(PROCESSING / "r76-82v-dcrc1pb.dataset.toml").spec.output)
manifest = json.loads((OUT / "_goldpan.json").read_text(encoding="utf-8"))
PRE = manifest["streams"]["NAI-S2"]["detector"]["segments"][0]["pretrigger_samples"]
print(f"{OUT}: {len(manifest['streams']['NAI-S2']['files'])} files in the manifest, baseline window = samples 10 to {PRE}")
CACHE = Path.home() / "goldsluice-output" / f"relcal-bstd-height-82v-p{PRE}.npz"
OLD_CACHE = Path.home() / "goldsluice-output" / "relcal-bstd-82v.npz"          # made by bstd-distribution.ipynb from the output with the 500 window

def field(tab, k, name):
    # one quantity of one channel: the channel is a struct column of the Parquet file
    return pc.struct_field(tab[f"Side2p{k}"].combine_chunks(), name).to_numpy(zero_copy_only=False)

if CACHE.exists():
    z = np.load(CACHE)
    series, trigger, bstd, height = z["series"], z["trigger"], z["bstd"], z["height"]
else:
    S_, T_, B_, H_ = [], [], [], []
    for f in sorted((OUT / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
        tab = pq.read_table(f, columns=["series", "trigger_type"] + [f"Side2p{k}" for k in LETTERS])
        S_.append(np.asarray(tab["series"].to_pylist())); T_.append(np.asarray(tab["trigger_type"].to_pylist()))
        B_.append(np.column_stack([field(tab, k, "baseline_std") for k in LETTERS]))
        H_.append(np.column_stack([field(tab, k, "current_max") for k in LETTERS]))
    series, trigger, bstd, height = np.concatenate(S_), np.concatenate(T_), np.vstack(B_), np.vstack(H_)
    np.savez(CACHE, series=series, trigger=trigger, bstd=bstd, height=height)
real = trigger == "Unknown"
rand = trigger == "Physics"
print(f"{len(series):,} events in {len(np.unique(series))} series: {real.sum():,} real triggers, {rand.sum():,} random triggers")

## The distribution, channel by channel

Real triggers in blue and random triggers in grey (each normalised), on logarithmic axes. The dotted line is 2 times the median `bstd` of the random triggers. The table gives the percentiles, and the share of the real triggers above that line.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
    r, q = bstd[real, i], bstd[rand, i]
    lo, hi = np.percentile(bstd[:, i], [0.01, 99.99])
    bins = np.geomspace(max(lo, 0.3), hi * 1.5, 160)
    ax.hist(q, bins=bins, histtype="stepfilled", color="0.8", density=True, label="random triggers")
    ax.hist(r, bins=bins, histtype="step", color="C0", density=True, lw=1.3, label="real triggers")
    ax.axvline(2 * np.median(q), color="C3", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(f"Side2p{k}", fontsize=11)
    ax.set_xlabel(f"bstd (counts), samples 10 to {PRE}")
    ax.set_ylabel("fraction of events per unit bstd")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{'':>12}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
for name, mask in [("real", real), ("random", rand)]:
    for p in (1, 50, 99, 99.9):
        print(f"{name + ' ' + str(p) + ' %':>12}" + "".join(f"{np.percentile(bstd[mask, i], p):>11.2f}" for i in range(6)))
print(f"{'2 x med.rnd':>12}" + "".join(f"{2 * np.median(bstd[rand, i]):>11.2f}" for i in range(6)))
print(f"{'real above':>12}" + "".join(f"{(bstd[real, i] > 2 * np.median(bstd[rand, i])).mean():>10.1%} " for i in range(6)))

## The old window against the new one

The real triggers with the window to sample 500 (the cache of `bstd-distribution.ipynb`) and to sample 450 (this output), for the same events. If the rising edge was the cause of the shift, the new distribution should sit lower, much closer to the random-trigger one. The last rows of the table give the medians and the share of events above twice the random-trigger median in both.

In [ ]:
if OLD_CACHE.exists():
    old = np.load(OLD_CACHE)
    same = len(old["series"]) == len(series) and (old["series"] == series).all() and (old["trigger"] == trigger).all()
    print("same events in the same order:", same)
else:
    old, same = None, False
    print("no cache of the 500 window: skipped")
if same:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
        lo, hi = np.percentile(bstd[:, i], [0.01, 99.99])
        bins = np.geomspace(max(lo, 0.3), hi * 1.5, 160)
        ax.hist(bstd[rand, i], bins=bins, histtype="stepfilled", color="0.85", density=True, label="random, 450")
        ax.hist(old["bstd"][real, i], bins=bins, histtype="step", color="C1", density=True, lw=1.2, label="real, window to 500")
        ax.hist(bstd[real, i], bins=bins, histtype="step", color="C0", density=True, lw=1.2, label="real, window to 450")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(f"Side2p{k}", fontsize=11)
        ax.set_xlabel("bstd (counts)"); ax.set_ylabel("fraction of events per unit bstd")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    print(f"{'':>22}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
    print(f"{'median, window 500':>22}" + "".join(f"{np.median(old['bstd'][real, i]):>11.2f}" for i in range(6)))
    print(f"{'median, window 450':>22}" + "".join(f"{np.median(bstd[real, i]):>11.2f}" for i in range(6)))
    print(f"{'random median, 450':>22}" + "".join(f"{np.median(bstd[rand, i]):>11.2f}" for i in range(6)))
    thr = [2 * np.median(bstd[rand, i]) for i in range(6)]
    print(f"{'above 2x random, 500':>22}" + "".join(f"{(old['bstd'][real, i] > thr[i]).mean():>10.1%} " for i in range(6)))
    print(f"{'above 2x random, 450':>22}" + "".join(f"{(bstd[real, i] > thr[i]).mean():>10.1%} " for i in range(6)))

## Height against `bstd`

The height of the pulse (`current_max`, counts above the baseline) against `bstd`, real triggers, per channel; the dotted line is 2 times the random-trigger median. A ridge in which `bstd` grows with the height would mean that the edge is still in the window.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
    bb = np.geomspace(max(np.percentile(bstd[:, i], 0.01), 0.5), np.percentile(bstd[:, i], 99.99), 100)
    hb = np.geomspace(3, 3000, 100)
    ax.hist2d(bstd[real, i], height[real, i].clip(1), bins=[bb, hb], norm=colors.LogNorm(), cmin=1, cmap="viridis")
    ax.axvline(2 * np.median(bstd[rand, i]), color="C3", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(f"bstd (counts), samples 10 to {PRE}"); ax.set_ylabel(f"Side2p{k}: height (counts)")
fig.tight_layout()
plt.show()

## By series

The median `bstd` of the real triggers of each series, per channel, for the new window; a series that stands out is a series whose baseline noise is different.

In [ ]:
sers = sorted(np.unique(series))
print(f"{'series':>14}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
for s in sers:
    m = real & (series == s)
    print(f"{s:>14}" + "".join(f"{np.median(bstd[m, i]):>11.2f}" for i in range(6)))

**What this shows, and what is not done.** The `bstd` of every channel with the baseline window ending at sample 450, against the old window and against the random triggers. Not done: a quality cut on `bstd`; the choice (a multiple of the random-trigger median, a percentile, or a limit per series) follows from these plots. The processing has no status file yet (that is `goldsluice run --replace` and the status file of pull request 201), so that the output was completely rewritten is known only from its 436 files in the manifest.